In [1]:
# 08_listwise_wj_distillation.ipynb
# Listwise WJ-to-cosine distillation for the MLP candidate generator.
#
# This is the KDIndex-style experiment:
#   teacher = exact WeightedJaccard scores over candidate lists
#   student = MLP cosine scores over the same lists
#   loss    = listwise KL divergence between teacher ranking distribution and student scores
#
# Start on 10k, then scale to full if it improves candidate recall.


In [2]:
import os
import pickle
import random
import time

import nmslib
import numpy as np
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

THREADS = 32
QUERY_START_10K = 8000
QUERY_START_FULL = 187019

class QuadtreeCompressorV1(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x):
        return self.net(x)

class QuadtreeCompressorV1Fixed(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x):
        x = torch.log1p(x * 1e6)
        return self.net(x)

def get_mem_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

def generate_embeddings(model, data, device, batch_size=512):
    model.eval()
    chunks = []
    with torch.no_grad():
        for start in tqdm(range(0, len(data), batch_size), desc="Embedding"):
            batch = torch.tensor(data[start:start + batch_size], dtype=torch.float32, device=device)
            chunks.append(F.normalize(model(batch), dim=1).cpu().numpy())
    return np.vstack(chunks)

def build_cosine_index(corpus_embs, ef_search=200):
    m0 = get_mem_mb()
    idx = nmslib.init(method="hnsw", space="cosinesimil")
    for i in tqdm(range(len(corpus_embs)), desc="Adding", mininterval=2.0):
        idx.addDataPoint(i, corpus_embs[i])
    t0 = time.time()
    idx.createIndex({"M": 20, "efConstruction": 200, "post": 1}, print_progress=True)
    build_s = time.time() - t0
    idx_mb = get_mem_mb() - m0
    idx.setQueryTimeParams({"efSearch": ef_search})
    return idx, build_s, idx_mb

def recall_at_k(gt_lookup, nbrs, query_start_id, k):
    total = 0.0
    count = 0
    for i, ids in enumerate(nbrs):
        gt = set(gt_lookup.get(query_start_id + i, [])[:k])
        if not gt:
            continue
        total += len(gt & set(ids[:k])) / len(gt)
        count += 1
    return total / count if count else 0.0


In [3]:
# Configuration
dataset_name = "10k"        # "10k" first, then "full"
start_variant = "harddist"   # "base" or "harddist"
device = torch.device("cuda:0")
seed = 123

pool_k = 500                 # cosine candidates to build each list
list_size = 64               # candidates per query used in listwise training
force_gt_top = 10            # inject top WJ positives into every list
max_queries = None           # set e.g. 1000 for a smoke test
val_frac = 0.1

last_layer_only = True
batch_size = 16
epochs = 8
lr = 5e-5
weight_decay = 1e-4
teacher_temp = 0.03
student_temp = 0.05
preserve_weight = 0.10

candidate_ks = [100, 200, 500]
rerank_batch_size = 16
out_path = "/tmp/results_listwise_wjdistill.pkl"

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


In [4]:
# Load data and starting checkpoint
if dataset_name == "10k":
    qt = np.load("/tmp/qt_10k.npy")
    with open("/tmp/gt_lookup_10k.pkl", "rb") as f:
        gt = pickle.load(f)
    query_start = QUERY_START_10K
    model_cls = QuadtreeCompressorV1
    base_ckpt = "/tmp/best_compressor_v1_clean.pt"
    hard_ckpt = "/tmp/best_compressor_hardneg_wjdistill_10k.pt"
    listwise_ckpt = "/tmp/best_compressor_listwise_wjdistill_10k.pt"
elif dataset_name == "full":
    qt = np.load("/tmp/qtree_vectors_full.npy")
    with open("/tmp/gt_lookup_full.pkl", "rb") as f:
        gt = pickle.load(f)
    query_start = QUERY_START_FULL
    model_cls = QuadtreeCompressorV1Fixed
    base_ckpt = "/tmp/best_compressor_full_fixed.pt"
    hard_ckpt = "/tmp/best_compressor_hardneg_wjdistill_full.pt"
    listwise_ckpt = "/tmp/best_compressor_listwise_wjdistill_full.pt"
else:
    raise ValueError(dataset_name)

start_ckpt = hard_ckpt if start_variant == "harddist" and os.path.exists(hard_ckpt) else base_ckpt
corpus_qt = qt[:query_start]
query_qt = qt[query_start:]
corpus_sums = corpus_qt.sum(axis=1)
query_ids = [qid for qid in sorted(gt) if query_start <= qid < len(qt)]
if max_queries is not None:
    query_ids = query_ids[:max_queries]

print(f"dataset={dataset_name} | start_ckpt={start_ckpt}")
print(f"corpus={corpus_qt.shape} | queries={query_qt.shape} | train queries={len(query_ids)}")


dataset=10k | start_ckpt=/tmp/best_compressor_hardneg_wjdistill_10k.pt
corpus=(8000, 18499) | queries=(2000, 18499) | train queries=1818


In [5]:
# Mine candidate lists with the starting MLP model.
start_model = model_cls(qt.shape[1], out_dim=512).to(device)
start_model.load_state_dict(torch.load(start_ckpt, weights_only=True, map_location=device))
start_model.eval()

embs = generate_embeddings(start_model, qt, device)
corpus_embs = embs[:query_start]
query_embs = embs[query_start:]
idx, build_s, idx_mb = build_cosine_index(corpus_embs)

local_query_indices = [qid - query_start for qid in query_ids]
print(f"Mining top-{pool_k} candidates...")
t0 = time.time()
mined_nbrs = idx.knnQueryBatch(query_embs[local_query_indices], k=pool_k, num_threads=THREADS)
print(f"Mining time={time.time() - t0:.2f}s")


Adding: 100%|██████████| 8000/8000 [00:00<00:00, 710793.57it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

Mining top-500 candidates...
Mining time=0.07s


In [6]:
# Build fixed-size candidate lists: WJ positives + hard cosine candidates.
list_query_ids = []
list_candidate_ids = []
for qid, (cand_ids, _) in tqdm(zip(query_ids, mined_nbrs), total=len(query_ids), desc="Building lists"):
    gt_pos = [pid for pid in gt.get(qid, []) if 0 <= pid < len(corpus_qt)]
    if not gt_pos:
        continue
    selected = []
    for pid in gt_pos[:force_gt_top]:
        if pid not in selected:
            selected.append(int(pid))
    for cid in cand_ids:
        cid = int(cid)
        if cid not in selected:
            selected.append(cid)
        if len(selected) >= list_size:
            break
    if len(selected) < list_size:
        continue
    list_query_ids.append(qid)
    list_candidate_ids.append(selected[:list_size])

list_query_ids = np.asarray(list_query_ids, dtype=np.int64)
list_candidate_ids = np.asarray(list_candidate_ids, dtype=np.int64)
print(f"listwise examples={len(list_query_ids):,} | shape={list_candidate_ids.shape}")


Building lists: 100%|██████████| 1818/1818 [00:00<00:00, 17088.72it/s]

listwise examples=1,818 | shape=(1818, 64)


In [7]:
# Compute teacher WJ scores for every query-candidate list on GPU.
def score_lists_wj_gpu(query_ids_np, cand_ids_np, qt, corpus_qt, corpus_sums, device, batch_size=16):
    corpus_t = torch.from_numpy(corpus_qt).to(device=device, dtype=torch.float32)
    corpus_sums_t = torch.from_numpy(corpus_sums).to(device=device, dtype=torch.float32)
    scores = np.empty(cand_ids_np.shape, dtype=np.float32)
    for start in tqdm(range(0, len(query_ids_np), batch_size), desc="Teacher WJ scores"):
        qids = query_ids_np[start:start + batch_size]
        cids = cand_ids_np[start:start + batch_size]
        q_np = qt[qids]
        q_t = torch.from_numpy(q_np).to(device=device, dtype=torch.float32)
        ids_t = torch.from_numpy(cids).to(device=device)
        c_t = corpus_t[ids_t]
        mins = torch.minimum(q_t[:, None, :], c_t).sum(dim=2)
        maxs = q_t.sum(dim=1, keepdim=True) + corpus_sums_t[ids_t] - mins
        scores[start:start + len(qids)] = (mins / maxs.clamp_min(1e-10)).cpu().numpy()
    del corpus_t, corpus_sums_t
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return scores

teacher_scores = score_lists_wj_gpu(list_query_ids, list_candidate_ids, qt, corpus_qt, corpus_sums, device, batch_size)
print("teacher score stats:", float(teacher_scores.min()), float(np.median(teacher_scores)), float(teacher_scores.max()))


Teacher WJ scores: 100%|██████████| 114/114 [00:00<00:00, 965.13it/s]

teacher score stats: 0.0003566670638974756 0.6860353946685791 0.9997305870056152


In [8]:
# Dataset and listwise loss
class ListwiseWJDataset(Dataset):
    def __init__(self, qt, corpus_qt, qids, cand_ids, scores):
        self.qt = qt
        self.corpus_qt = corpus_qt
        self.qids = qids
        self.cand_ids = cand_ids
        self.scores = scores
    def __len__(self):
        return len(self.qids)
    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.qt[self.qids[idx]]).float(),
            torch.from_numpy(self.corpus_qt[self.cand_ids[idx]]).float(),
            torch.from_numpy(self.scores[idx]).float(),
        )

perm = np.random.default_rng(seed).permutation(len(list_query_ids))
val_n = max(1, int(len(perm) * val_frac))
val_idx = perm[:val_n]
train_idx = perm[val_n:]

train_ds = ListwiseWJDataset(qt, corpus_qt, list_query_ids[train_idx], list_candidate_ids[train_idx], teacher_scores[train_idx])
val_ds = ListwiseWJDataset(qt, corpus_qt, list_query_ids[val_idx], list_candidate_ids[val_idx], teacher_scores[val_idx])
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

teacher_model = model_cls(qt.shape[1], out_dim=512).to(device)
teacher_model.load_state_dict(torch.load(start_ckpt, weights_only=True, map_location=device))
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad = False

def set_trainable_scope(model, last_layer_only=True):
    if not last_layer_only:
        for p in model.parameters():
            p.requires_grad = True
        return
    for p in model.parameters():
        p.requires_grad = False
    for layer_idx in (6, 7):
        for p in model.net[layer_idx].parameters():
            p.requires_grad = True

def listwise_loss(model, teacher_model, q, cands, wj_scores):
    b, l, d = cands.shape
    zq = F.normalize(model(q), dim=1)
    zc = F.normalize(model(cands.reshape(b * l, d)).reshape(b, l, -1), dim=2)
    student_scores = torch.einsum("bd,bld->bl", zq, zc)

    teacher_probs = F.softmax(wj_scores / teacher_temp, dim=1)
    student_log_probs = F.log_softmax(student_scores / student_temp, dim=1)
    kl = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean") * (student_temp ** 2)

    with torch.no_grad():
        tq = F.normalize(teacher_model(q), dim=1)
    preserve = F.mse_loss(zq, tq)
    return kl + preserve_weight * preserve, kl.detach(), preserve.detach()


In [9]:
# Train listwise-distilled model
model = model_cls(qt.shape[1], out_dim=512).to(device)
model.load_state_dict(torch.load(start_ckpt, weights_only=True, map_location=device))
set_trainable_scope(model, last_layer_only=last_layer_only)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
best_val = float("inf")
history = []

for epoch in range(1, epochs + 1):
    model.train()
    losses = []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{epochs} train", leave=False)
    for q, cands, scores in pbar:
        q = q.to(device, non_blocking=True)
        cands = cands.to(device, non_blocking=True)
        scores = scores.to(device, non_blocking=True)
        loss, kl, preserve = listwise_loss(model, teacher_model, q, cands, scores)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
        pbar.set_postfix(loss=f"{losses[-1]:.5f}", kl=f"{float(kl):.5f}", preserve=f"{float(preserve):.5f}")

    model.eval()
    val_losses = []
    with torch.no_grad():
        for q, cands, scores in val_loader:
            q = q.to(device, non_blocking=True)
            cands = cands.to(device, non_blocking=True)
            scores = scores.to(device, non_blocking=True)
            loss, kl, preserve = listwise_loss(model, teacher_model, q, cands, scores)
            val_losses.append(float(loss.detach().cpu()))
    train_loss = float(np.mean(losses))
    val_loss = float(np.mean(val_losses))
    scheduler.step()
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), listwise_ckpt)
    print(f"Epoch {epoch:02d} | train={train_loss:.6f} | val={val_loss:.6f} | best={best_val:.6f}")

print(f"Done. Best checkpoint: {listwise_ckpt}")


Trainable params: 525,312 / 80,501,760


Epoch 01 | train=0.003074 | val=0.003256 | best=0.003256


Epoch 02 | train=0.003075 | val=0.003055 | best=0.003055


Epoch 03 | train=0.003027 | val=0.003778 | best=0.003055


Epoch 04 | train=0.002987 | val=0.003538 | best=0.003055


Epoch 05 | train=0.002963 | val=0.003172 | best=0.003055


Epoch 06 | train=0.002984 | val=0.003027 | best=0.003027


Epoch 07 | train=0.002980 | val=0.002951 | best=0.002951


Epoch 08 | train=0.002954 | val=0.003734 | best=0.002951
Done. Best checkpoint: /tmp/best_compressor_listwise_wjdistill_10k.pt


In [10]:
# Evaluate base/start vs listwise model with two-stage GPU WJ rerank.
def rerank_wj_gpu(query_qt, nbrs_raw, corpus_qt, corpus_sums, device, batch_size=16):
    corpus_t = torch.from_numpy(corpus_qt).to(device=device, dtype=torch.float32)
    corpus_sums_t = torch.from_numpy(corpus_sums).to(device=device, dtype=torch.float32)
    reranked = [None] * len(nbrs_raw)
    for start in tqdm(range(0, len(nbrs_raw), batch_size), desc="GPU WJ rerank"):
        batch = nbrs_raw[start:start + batch_size]
        groups = {}
        for offset, (ids, _) in enumerate(batch):
            ids_arr = np.asarray(ids, dtype=np.int64)
            groups.setdefault(len(ids_arr), []).append((start + offset, ids_arr))
        for _, items in groups.items():
            ids_np = np.stack([ids for _, ids in items], axis=0)
            query_np = np.stack([query_qt[absolute_i] for absolute_i, _ in items], axis=0)
            ids_t = torch.from_numpy(ids_np).to(device=device)
            q_t = torch.from_numpy(query_np).to(device=device, dtype=torch.float32)
            c_t = corpus_t[ids_t]
            mins = torch.minimum(q_t[:, None, :], c_t).sum(dim=2)
            maxs = q_t.sum(dim=1, keepdim=True) + corpus_sums_t[ids_t] - mins
            order = torch.argsort(mins / maxs.clamp_min(1e-10), dim=1, descending=True).cpu().numpy()
            for row, (absolute_i, ids) in zip(order, items):
                reranked[absolute_i] = ids[row].tolist()
    del corpus_t, corpus_sums_t
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return reranked

def evaluate_two_stage(eval_model, label):
    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)
    embs_eval = generate_embeddings(eval_model, qt, device)
    corpus_eval = embs_eval[:query_start]
    query_eval = embs_eval[query_start:]
    idx_eval, build_eval_s, idx_eval_mb = build_cosine_index(corpus_eval)
    results = {}
    for k in candidate_ks:
        t0 = time.time()
        nbrs = idx_eval.knnQueryBatch(query_eval, k=k, num_threads=THREADS)
        hnsw_s = time.time() - t0
        t0 = time.time()
        reranked = rerank_wj_gpu(query_qt, nbrs, corpus_qt, corpus_sums, device, rerank_batch_size)
        rerank_s = time.time() - t0
        qps = len(query_eval) / (hnsw_s + rerank_s)
        res = {
            10: recall_at_k(gt, reranked, query_start, 10),
            50: recall_at_k(gt, reranked, query_start, 50),
            100: recall_at_k(gt, reranked, query_start, 100),
            500: recall_at_k(gt, reranked, query_start, 500) if k >= 500 else np.nan,
            "qps": qps,
            "hnsw_s": hnsw_s,
            "rerank_s": rerank_s,
            "build_s": build_eval_s,
            "idx_mb": idx_eval_mb,
        }
        results[f"k{k}_wj_rerank"] = res
        print(f"K={k} | R@100={res[100]:.4f} | R@500={res[500] if res[500] == res[500] else float('nan'):.4f} | QPS={qps:.1f}")
    return results

start_eval_results = evaluate_two_stage(start_model, "Start MLP + cosine candidates + GPU WJ rerank")
listwise_model = model_cls(qt.shape[1], out_dim=512).to(device)
listwise_model.load_state_dict(torch.load(listwise_ckpt, weights_only=True, map_location=device))
listwise_eval_results = evaluate_two_stage(listwise_model, "Listwise WJ-distilled MLP + cosine candidates + GPU WJ rerank")



Start MLP + cosine candidates + GPU WJ rerank


Adding: 100%|██████████| 8000/8000 [00:00<00:00, 689710.83it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
GPU WJ rerank: 100%|██████████| 125/125 [00:00<00:00, 868.94it/s]


K=100 | R@100=0.8641 | R@500=nan | QPS=9398.7


GPU WJ rerank: 100%|██████████| 125/125 [00:00<00:00, 639.94it/s]


K=200 | R@100=0.9862 | R@500=nan | QPS=7259.4


...

K=100 | R@100=0.8506 | R@500=nan | QPS=10689.6


GPU WJ rerank: 100%|██████████| 125/125 [00:00<00:00, 649.94it/s]


K=200 | R@100=0.9822 | R@500=nan | QPS=7556.1


GPU WJ rerank: 100%|██████████| 125/125 [00:00<00:00, 305.10it/s]


K=500 | R@100=0.9989 | R@500=0.9611 | QPS=3904.5


In [11]:
# Save and summarize results.
run_key = time.strftime(f"{dataset_name}_listwise_%Y%m%d_%H%M%S")
record = {
    "config": {
        "dataset_name": dataset_name,
        "start_variant": start_variant,
        "start_ckpt": start_ckpt,
        "pool_k": pool_k,
        "list_size": list_size,
        "force_gt_top": force_gt_top,
        "last_layer_only": last_layer_only,
        "epochs": epochs,
        "lr": lr,
        "teacher_temp": teacher_temp,
        "student_temp": student_temp,
        "preserve_weight": preserve_weight,
        "candidate_ks": candidate_ks,
    },
    "history": history,
    "start_eval": start_eval_results,
    "listwise_eval": listwise_eval_results,
}
try:
    with open(out_path, "rb") as f:
        saved = pickle.load(f)
except FileNotFoundError:
    saved = {"runs": {}}
saved.setdefault("runs", {})[run_key] = record
with open(out_path, "wb") as f:
    pickle.dump(saved, f)
print(f"Saved run {run_key} to {out_path}")

print("\n" + "=" * 96)
print("START VS LISTWISE-DISTILLED TWO-STAGE RESULTS")
print("=" * 96)
print(f"{'Model':<12} {'Method':<18} {'R@10':>7} {'R@50':>7} {'R@100':>7} {'R@500':>7} {'QPS':>9}")
print("-" * 96)
for label, results in [("Start", start_eval_results), ("Listwise", listwise_eval_results)]:
    for method, res in results.items():
        def fmt(k):
            return f"{res[k]:>7.4f}" if res[k] == res[k] else f"{'-':>7}"
        print(f"{label:<12} {method:<18} {fmt(10)} {fmt(50)} {fmt(100)} {fmt(500)} {res['qps']:>9.1f}")


Saved run 10k_listwise_20260427_134917 to /tmp/results_listwise_wjdistill.pkl

START VS LISTWISE-DISTILLED TWO-STAGE RESULTS
Model        Method                R@10    R@50   R@100   R@500       QPS
------------------------------------------------------------------------------------------------
Start        k100_wj_rerank      0.9962  0.9724  0.8641       -    9398.7
Start        k200_wj_rerank      0.9966  0.9974  0.9862       -    7259.4
Start        k500_wj_rerank      0.9966  0.9986  0.9989  0.9679    3811.1
Listwise     k100_wj_rerank      0.9963  0.9674  0.8506       -   10689.6
Listwise     k200_wj_rerank      0.9965  0.9968  0.9822       -    7556.1
Listwise     k500_wj_rerank      0.9966  0.9986  0.9989  0.9611    3904.5
